In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import torch
import gymnasium as gym
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss
from neuralforecast.models import TimesNet
from stable_baselines3 import A2C, PPO
from gymnasium.utils.env_checker import check_env

from config.config import appConfig
from src.data_prep import data_prep, data_for_timesnet
from src.portfolio_env import PortfolioEnv

/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-09 04:44:56,352	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-09 04:44:57,027	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
config = appConfig(
    initial_balance=100000,
    allow_short_selling=False,
    technical_indicator_list=['MACD','RSI_14','CCI_20','SMA_20','EMA_20'], #'ADX_14',
    transaction_cost=0.001,
    tickers=["BTC-USD","ETH-USD","LTC-USD","LINK-USD","BCH-USD","UNI-USD","XLM-USD","FIL-USD","BNB-USD","SOL-USD","XRP-USD","ADA-USD","SHIB-USD","TON-USD","DOGE-USD","AVAX-USD","TRX-USD","DOT-USD","MATIC-USD","ETC-USD"],
    train_starting_date="2020-09-22",
    train_ending_date="2022-06-07",
    test_starting_date="2022-06-08",
    test_ending_date="2023-01-03",
    THRESHOLD_PARAMETER=0.015,
    oc_upper_threshold=0.01, oc_lower_threshold=-0.01, oc_k_loss=0.1, oc_k_gain=0.1, oc_n=0.725,
    ra_upper_threshold=0.01, ra_lower_threshold=-0.01, ra_k_loss=0.1, ra_k_gain=0.1, ra_n=1.22,
)


In [ ]:
train_df = data_prep(config.train_starting_date, config.train_ending_date, config.ticker_list)
test_df = data_prep(config.test_starting_date,  config.test_ending_date,  config.ticker_list)
print(f"Train shape: {train_df.shape}, dates: {train_df['ds'].nunique()}, tickers: {train_df['unique_id'].nunique()}")
print(f"Test  shape: {test_df.shape},  dates: {test_df['ds'].nunique()},  tickers: {test_df['unique_id'].nunique()}")


[*********************100%***********************]  20 of 20 completed
/mnt/c/Users/akgup/Documents/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()
[*********************100%***********************]  19 of 20 completed


Train shape: (7960, 13), dates: 398, tickers: 20
Test  shape: (3680, 13),  dates: 184,  tickers: 20


/mnt/c/Users/akgup/Documents/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()


In [ ]:
model = TimesNet(h=1,
                input_size=24, #Context Window
                hidden_size = 16,
                conv_hidden_size = 32,
                loss=DistributionLoss(distribution='Normal', level=[80, 90]),
                scaler_type='standard',
                learning_rate=1e-3,
                max_steps=100,
                val_check_steps=50,
                early_stop_patience_steps=2)
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=data_for_timesnet(train_df),val_size=1)

Seed set to 1
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | loss           | DistributionLoss | 5      | train
1 | padder_train   | ConstantPad1d    | 0      | train
2 | scaler         | TemporalNorm     | 0      | train
3 | model          | ModuleList       | 586 K  | train
4 | enc_embedding  | DataEmbedding    | 48     | train
5 | layer_norm     | LayerNorm        | 32     | train
6 | predict_linear | Linear           | 625    | train
7 | projection     | Linear           | 34     | train
------------------------------------------------------------
587 K     Trainable params
5         Non-trainable params
587 K     Total params
2.348     Total estimated model params size (MB)
50        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 99: 100%|██████████| 1/1 [00:01<00:00,  0.51it/s, v_num=16, train_loss_step=0.710, train_loss_epoch=0.710, valid_loss=-0.532] 

`Trainer.fit` stopped: `max_steps=100` reached.


Epoch 99: 100%|██████████| 1/1 [00:02<00:00,  0.50it/s, v_num=16, train_loss_step=0.710, train_loss_epoch=0.710, valid_loss=-0.532]


In [ ]:
gym.register(
    id='PortfolioEnv-v4',
    entry_point=PortfolioEnv,
    kwargs={'data': train_df, 'config': config}
)
env = gym.make('PortfolioEnv-v4')
try:
    check_env(env.unwrapped)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")


In [ ]:
model_rl = A2C('MlpPolicy', env, verbose=1)
model_rl.learn(total_timesteps=10000)
print("Training completed!")

In [ ]:
date = "2023-01-02"
ticker = "BTC-USD"

In [ ]:
forecast = nf.predict(train_df)

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:02<00:00,  0.40it/s]


In [ ]:
forecast